In [1]:
import pandas as pd
import numpy as np
import os
import glob
from types import SimpleNamespace
from scipy import signal

In [ ]:
# 處理數據

# 把時間點打散鋪平
def resample_time(df):
    new_rows = []
    # 1. 找出分組點：當時間差大於 0.01 秒時，視為新的一組
    # 計算與上一筆的時間差
    df['diff'] = df['time'].diff().fillna(0)
    group_indices = df[df['diff'] > 0.001].index.tolist()
    group_indices = [0] + group_indices + [len(df)]

    # 2. 遍歷每一組進行時間平攤
    for i in range(len(group_indices) - 1):
        start_idx = group_indices[i]
        end_idx = group_indices[i+1]
        
        group = df.iloc[start_idx:end_idx].copy()
        count = len(group)
        
        if count == 0: continue
        
        # 取得當前組的起始時間
        t_start = group.iloc[0]['time']
        
        # 取得下一組的起始時間（如果是最後一組，則按最後兩筆的間隔推算）
        if end_idx < len(df):
            t_next = df.iloc[end_idx]['time']
        else:
            # 最後一組的結束時間假設增加一個步長
            t_next = t_start + 0.031  # 這裡的 0.031 是參考你截圖的平均間隔
            
        # 3. 均勻計算時間間隔
        # 在 t_start 和 t_next 之間均勻產生 count 個時間點
        new_times = np.linspace(t_start, t_next, count, endpoint=False).round(3)
        group['time'] = new_times
        new_rows.append(group)

    return pd.concat(new_rows).drop(columns=['diff'])


# 攤平、內插成100hz
def to_100hz(df):
    # 1. 定義目標時間軸：從 0 開始，每 0.01 秒一格，直到原始數據結束
    t_min = 0.00
    t_max = df['time'].max()
    # 產生 [0.00, 0.01, 0.02, ..., t_max]
    new_time_axis = np.arange(t_min, t_max + 0.01, 0.01).round(2)
    
    # 2. 準備存儲新數據的字典
    resampled_data = {'time': new_time_axis}
    resampled_data['sensor'] = df['sensor'].iloc[0]
    
    # 3. 對每一個數值欄位進行線性內插
    # numpy.interp(目標點, 原始時間, 原始數值)
    for col in df.columns:
        if col not in ['time', 'sensor']:
            interp_values = np.interp(new_time_axis, df['time'], df[col])
            if 'θ' in col: 
                resampled_data[col] = np.round(interp_values, 2)
            else: # 加速度 (a) 與 角速度 (ω)
                resampled_data[col] = np.round(interp_values, 3)
            
    return pd.DataFrame(resampled_data)



def get_Rx(θX):
    return np.array([[1, 0, 0], 
                   [0, np.cos(θX), np.sin(θX)], 
                   [0, -np.sin(θX), np.cos(θX)]])

def get_Ry(θY):
    return np.array([[np.cos(θY), 0, -np.sin(θY)], 
                   [0, 1, 0], 
                   [np.sin(θY), 0, np.cos(θY)]])

def get_Rz(θZ):
    return np.array([[np.cos(θZ), np.sin(θZ), 0], 
                   [-np.sin(θZ), np.cos(θZ), 0], 
                   [0, 0, 1]])


# 扣g然後投影到朝前座標系
def _front_and_g_out(df, t_start, t_end):
    new_rows = []

    mask = (df['time'] >= t_start) & (df['time'] <= t_end)
    df_filtered = df.loc[mask]

    for idx, row in df_filtered.iterrows():
        θX = np.radians(row['θX(°)'])
        θY = np.radians(row['θY(°)'])
        a = np.array([row['aX(g)'], row['aY(g)'], row['aZ(g)']])
        ω = np.array([row['ωX(°/s)'], row['ωY(°/s)'], row['ωZ(°/s)']])

        invRx, invRy = get_Rx(-θX), get_Ry(-θY)
        Ryx = invRy @ invRx
        
        yp = invRx[1, :]
        z = Ryx[2, :]

        xt = np.cross(yp, z)
        yt = yp
        zt = z

        Rt = np.array([xt, yt, zt])
        
        a_proj = Rt @ a
        ω_proj = Rt @ ω

        new_data = {
            'time': np.round(row['time'] - t_start, 2),
            'sensor': row['sensor'],
            'aX(g)': np.round(a_proj[0], 3),
            'aY(g)': np.round(a_proj[1], 3),
            'aZ(g)': np.round(a_proj[2] - 1, 3),  # 扣掉重力加速度
            'ωX(°/s)': np.round(ω_proj[0], 3),
            'ωY(°/s)': np.round(ω_proj[1], 3),
            'ωZ(°/s)': np.round(ω_proj[2], 3),
        }

        new_rows.append(new_data)

    return pd.DataFrame(new_rows)


# 把腰的數據扣掉然後併成一個檔案
def get_final(folder_path):  
    # --- [步驟 A] 手動定義所有感測器 ID ---
    sensors = ['FD', '5D', 'C8', 'B1', 'D9']
    waist_id = 'FD'
    
    # --- [步驟 B] 讀取所有檔案並存入 dictionary ---
    data_dict = {}
    
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            # 檢查檔案名稱結尾
            if file.endswith("_front_and_g_out.csv"):
                full_path = os.path.join(root, file)
                for s in sensors:
                    if s == waist_id:
                        output_path = full_path.replace('FD_front_and_g_out.csv', 'final.csv')
                        
                    if s in file:
                        data_dict[s] = pd.read_csv(full_path)
                       

    # --- [步驟 C] 定義所有的 Columns 名稱 ---
    # 我們先建立一個清單來存放 Header
    headers = ['time']
    
    # 1. 放入所有感測器的原始投影數據
    for s in sensors:
        if s == waist_id:
            headers += [f'{s}_aX(g)', f'{s}_aY(g)']
        else:
            headers += [f'{s}_aX(g)', f'{s}_aY(g)', f'{s}_aZ(g)', f'{s}_ωX(°/s)', f'{s}_ωY(°/s)', f'{s}_ωZ(°/s)']

    # --- [步驟 D] 逐行提取數據並合成 ---
    # 以腰部 (FD) 的時間軸為基準
    times = data_dict[waist_id]['time'].values
    final_rows = []

    for t in times:
        # 這一行要存入的所有數值
        current_row = {'time': t}

        w_row = data_dict[waist_id][data_dict[waist_id]['time'] == t]
        waX = w_row['aX(g)'].values[0]
        waY = w_row['aY(g)'].values[0]
        waZ = w_row['aZ(g)'].values[0]
        wωX = w_row['ωX(°/s)'].values[0]
        wωY = w_row['ωY(°/s)'].values[0]
        wωZ = w_row['ωZ(°/s)'].values[0]
        
        # 1. 抓取每個感測器在時間 t 的數值
        for s in sensors:
            # 找到該感測器在時間 t 的那一列
            row_s = data_dict[s][data_dict[s]['time'] == t]

            if s == waist_id:
                current_row[f'{s}_aX(g)'] = waX
                current_row[f'{s}_aY(g)'] = waY
                
            else:
                ax, ay, az = row_s['aX(g)'].values[0], row_s['aY(g)'].values[0], row_s['aZ(g)'].values[0]
                ωX, ωY, ωZ = row_s['ωX(°/s)'].values[0], row_s['ωY(°/s)'].values[0], row_s['ωZ(°/s)'].values[0]
                
                # 填入對應參數
                current_row[f'{s}_aX(g)'] = round(ax - waX, 3)
                current_row[f'{s}_aY(g)'] = round(ay - waY, 3)
                current_row[f'{s}_aZ(g)'] = round(az - waZ, 3)
                current_row[f'{s}_ωX(°/s)'] = round(ωX - wωX, 3)
                current_row[f'{s}_ωY(°/s)'] = round(ωY - wωY, 3)
                current_row[f'{s}_ωZ(°/s)'] = round(ωZ - wωZ, 3)

        final_rows.append(current_row)    
            

    # --- [步驟 E] 轉成 DataFrame 並儲存 ---
    df_final = pd.DataFrame(final_rows, columns=headers)
    df_final.to_csv(output_path, index=False)


def remove_spikes_and_smooth(data, kernel_size, cutoff, fs):
    # 1. 中值濾波：專門對付尖刺
    # kernel_size 必須是奇數，代表視窗大小
    med_data = signal.medfilt(data, kernel_size=kernel_size)
    
    # 2. 巴特沃斯低通濾波：讓曲線變漂亮
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(5, normal_cutoff, btype='low')
    final_data = signal.filtfilt(b, a, med_data)
    
    return final_data


# 滑動平均+濾波
def get_final2(file_path):
    df = pd.read_csv(file_path)
    
    fs = 100

    # 3. 對除了 time 以外的所有欄位進行濾波
    cols_to_filter = [c for c in df.columns if c != 'time']
    
    for col in cols_to_filter:
        df[col] = remove_spikes_and_smooth(df[col], kernel_size=5, cutoff=20, fs=fs)

    # 4. 去掉開頭 5 秒和結尾 5 秒
    # 我們根據時間數值來裁切，這樣比算 index 更安全
    t_start = 5.0
    t_end = 105.0  # 假設原始數據長度至少 110 秒，這樣裁切後還有 100 秒的數據
    
    df_cropped = df[(df['time'] >= t_start) & (df['time'] <= t_end)].copy()
    
    # 5. 重整時間軸（讓裁切後的起始時間變回 0）
    df_cropped['time'] = (df_cropped['time'] - 5.0).round(2)

    for col in df_cropped.columns:
        if col != 'time': 
            df_cropped[col] = df_cropped[col].round(3)
    
    return df_cropped


# 用成模型訓練用的數據
def process_ml_data():
    # 1. 設定路徑與參數
    root_path = r'C:\Users\User\Desktop\高三專題\數據' 
    save_dir = r'C:\Users\User\Desktop\高三專題\數據\ML\data' 

    # 參數設定
    window_sec = 0.5
    stride_sec = 0.05
    waist_id = 'FD'
    other_sensors = ['5D', 'C8', 'B1', 'D9']
    
    # 搜尋所有 final2 檔案
    files = glob.glob(os.path.join(root_path, "**/*final2.csv"), recursive=True)

    for file_path in files:
        df = pd.read_csv(file_path)

        experiment_name = os.path.basename(file_path).replace('.final2.csv', '')
        
        # --- [步驟 A] 角速度除以 100 ---
        ω_cols = [c for c in df.columns if '_ω' in c]
        df[ω_cols] = df[ω_cols] / 100.0

        # --- [步驟 B] 準備滑動視窗 ---
        # 自動計算 fs (假設 100Hz)
        fs = 100
        window_len = int(window_sec * fs)
        stride_len = int(stride_sec * fs)
        
        # 找出四個感測器的所有欄位 (Feature)
        feature_cols = []
        for s in other_sensors:
            feature_cols += [c for c in df.columns if s in c]
        
        # 找出腰部感測器的加速度欄位 (Label)
        waist_accel_cols = [f'{waist_id}_aX(g)', f'{waist_id}_aY(g)']

        X_list = []
        y_list = []

        # 開始切窗
        for start in range(0, len(df) - window_len, stride_len):
            end = start + window_len
            
            # 特徵：這 0.5 秒內四個感測器的所有數據
            window_data = df.iloc[start:end][feature_cols].values
            
            # 標籤下一刻第一筆的腰部加速度值
            last_waist_accel = df.iloc[end][waist_accel_cols].values
            
            X_list.append(window_data)
            y_list.append(last_waist_accel)

        # 轉成 NumPy Array
        X_array = np.array(X_list) # 維度: (視窗數, 50, 欄位數)
        y_array = np.array(y_list) # 維度: (視窗數, 2)

        # --- [步驟 C] 儲存結果 ---
        base_name = os.path.basename(file_path).replace('.csv', '')
        np.save(os.path.join(save_dir, f"{experiment_name}.X.npy"), X_array)
        np.save(os.path.join(save_dir, f"{experiment_name}.y.npy"), y_array)


def process_ml_data_linear():
    # 1. 設定路徑與參數
    root_path = r'C:\Users\User\Desktop\高三專題\數據' 
    save_dir = r'C:\Users\User\Desktop\高三專題\數據\ML\data' 

    # 為了對齊原資料的時間點，這些參數必須跟原本一模一樣
    window_sec = 0.5
    stride_sec = 0.05
    waist_id = 'FD'
    other_sensors = ['5D', 'C8', 'B1', 'D9']
    
    files = glob.glob(os.path.join(root_path, "**/*final2.csv"), recursive=True)

    for file_path in files:
        df = pd.read_csv(file_path)
        experiment_name = os.path.basename(file_path).replace('.final2.csv', '')
        
        # 角速度除以 100
        ω_cols = [c for c in df.columns if '_ω' in c]
        df[ω_cols] = df[ω_cols] / 100.0

        fs = 100
        window_len = int(window_sec * fs)
        stride_len = int(stride_sec * fs)
        
        feature_cols = []
        for s in other_sensors:
            feature_cols += [c for c in df.columns if s in c]
        
        waist_accel_cols = [f'{waist_id}_aX(g)', f'{waist_id}_aY(g)']

        X_list = []
        y_list = []

        # 這裡的迴圈範圍跟原本一模一樣，確保切出來的筆數跟 y 完全相同
        for start in range(0, len(df) - window_len, stride_len):
            end = start + window_len
            
            # 🔥 關鍵修改：不切區間！只取「當下」那一個時間點 (end-1) 的 24 個數值
            instant_data = df.iloc[end-1][feature_cols].values
            last_waist_accel = df.iloc[end][waist_accel_cols].values
            
            X_list.append(instant_data)
            y_list.append(last_waist_accel)

        X_array = np.array(X_list) # 維度: (視窗數, 24)
        y_array = np.array(y_list)# 維度: (視窗數, 2)

        # 儲存為專屬後綴，避免覆蓋掉原本的資料
        np.save(os.path.join(save_dir, f"{experiment_name}.X_linear.npy"), X_array)
        np.save(os.path.join(save_dir, f"{experiment_name}.y_linear.npy"), y_array)
        print(f"已生成 {experiment_name}: X={X_array.shape}, y={y_array.shape}")



# 試的

In [6]:
file_path = r"C:\Users\User\Desktop\高三專題\數據\straight\2\FD\s2.final2.csv"
experiment_name = os.path.basename(file_path).replace('.final2.csv', '')
experiment_name

's2'

In [6]:
folder_path = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise"
sensors = ['FD', '5D', 'C8', 'B1', 'D9']

for root, dirs, files in os.walk(folder_path):
        for file in files:
            # 檢查檔案名稱結尾
            if file.endswith("_front_and_g_out.csv"):
                full_path = os.path.join(root, file)
                for s in sensors:
                    if s in file:
                        print(full_path)

C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\5D\ba.5D_front_and_g_out.csv
C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\B1\ba.B1_front_and_g_out.csv
C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\C8\ba.C8_front_and_g_out.csv
C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\D9\ba.D9_front_and_g_out.csv
C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\FD\ba.FD_front_and_g_out.csv


In [7]:
θX = 100.85
Rx = get_Rx(θX)
a = Rx[2, :]
b = np.array([0, 0, 1])
c = np.array([0, 1, 0])
d = np.cross(b, c)
d
np.cos(np.radians(100.85))

np.float64(-0.1882384504636922)

In [8]:
f = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise"
for root, dirs, files in os.walk(f):
        for file in files:
            if file.endswith("_g_out.csv"):
                s = 'FD'
                if f".{s}_" in file:
                    df_waist = pd.read_csv(os.path.join(root, file))
                    print(f"已讀取 Sensor {s}: {file}")

new_filename = os.path.join(root, file.replace("_g_out.csv", "_standard.csv"))
new_filename                    

已讀取 Sensor FD: ba.FD_front_and_g_out.csv


'C:\\Users\\User\\Desktop\\高三專題\\數據\\bend\\anticlockwise\\FD\\ba.final.csv'

In [9]:
# 試攤平時間

f = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\5D\ba.5D.csv"
new_filename = f.replace('.csv', '_resampled.csv')
df = pd.read_csv(f, encoding='utf-8-sig')
df_resemple = resample_time(df)
df_resemple.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [10]:
# 試用100Hz

f = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\5D\ba.5D_resampled.csv"
new_filename = f.replace('_resampled.csv', '_100hz.csv')
df = pd.read_csv(f, encoding='utf-8-sig')
df_100hz = to_100hz(df)
df_100hz.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [11]:
# 試算靜止平均

f = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\5D\ba.5D_100hz.csv"
df = pd.read_csv(f, encoding='utf-8-sig')
data = get_initial_values(df, 0.00, 7.5)
print(f"{data.aX}", f"{data.aY}", f"{data.aZ}", f"{data.θX}", f"{data.θY}", f"{data.θZ}")


0.001 -0.007 1.025 -2.05 -1.68 -0.53


In [12]:
# 試扣g

f = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\5D\ba.5D_100hz.csv"
new_filename = f.replace('_100hz.csv', '_g_out.csv')
df = pd.read_csv(f, encoding='utf-8-sig')
initial_stats = get_initial_values(df, 0.00, 7.5)
df_g_out = g_out(df, 0.00, 7.5, 30, 90)
df_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [13]:
# 試創5個g_out.csv

sensors = ['FD', '5D', 'B1', 'C8', 'D9']
for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\{sensor}\ba.{sensor}.csv"
    new_filename = f.replace('.csv', '_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_g_out = g_out(to_100hz(resample_time(df)), 2.5, 7.5, 30, 85)
    df_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [14]:
# 試創5個front_and_g_out.csv

sensors = ['FD', '5D', 'B1', 'C8', 'D9']
for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\{sensor}\ba.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 10, 80)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')


KeyboardInterrupt: 

In [ ]:
# 試創一個final.csv

folder_path = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise"
get_final(folder_path)

In [15]:
# 創所有front_and_g_out.csv

sensors = ['FD', '5D', 'B1', 'C8', 'D9']
for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\{sensor}\ba.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 5, 115)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')

for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\bend\clockwise\{sensor}\bc.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 5, 115)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')  

for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\rightangle\anticlockwise\{sensor}\ra.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 15, 125)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')    

for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\rightangle\clockwise\{sensor}\rc.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 15, 125)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')      

for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\straight\1\{sensor}\s1.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 15, 125)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')    

for sensor in sensors:
    f = rf"C:\Users\User\Desktop\高三專題\數據\straight\2\{sensor}\s2.{sensor}.csv"
    new_filename = f.replace('.csv', '_front_and_g_out.csv')
    df = pd.read_csv(f, encoding='utf-8-sig')
    df_front_and_g_out = _front_and_g_out(to_100hz(resample_time(df)), 15, 125)
    df_front_and_g_out.to_csv(new_filename, index=False, encoding='utf-8-sig')     
         

In [16]:
# 創所有final.csv

file_of_folder_path = [r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise", 
                       r"C:\Users\User\Desktop\高三專題\數據\bend\clockwise", 
                       r"C:\Users\User\Desktop\高三專題\數據\rightangle\anticlockwise", 
                       r"C:\Users\User\Desktop\高三專題\數據\rightangle\clockwise", 
                       r"C:\Users\User\Desktop\高三專題\數據\straight\1", 
                       r"C:\Users\User\Desktop\高三專題\數據\straight\2"] 

for folder_path in file_of_folder_path:
    get_final(folder_path)  
    

In [22]:
# 試濾波並裁切(創一個final2.csv)

file_path = r"C:\Users\User\Desktop\高三專題\數據\bend\anticlockwise\FD\ba.final.csv"

filtered_cropped_df = get_final2(file_path)
new_filename = file_path.replace('.csv', '2.csv')
filtered_cropped_df.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [17]:
# 創所有final2.csv

files = glob.glob(os.path.join(r"C:\Users\User\Desktop\高三專題\數據", "**/*final.csv"), recursive=True)
for file in files:
    filtered_cropped_df = get_final2(file)
    new_filename = file.replace('.csv', '2.csv')
    filtered_cropped_df.to_csv(new_filename, index=False, encoding='utf-8-sig')

In [18]:
# 創所有ML_data

process_ml_data()

In [ ]:


experiment_name_list = ['ba', 'bc', 'ra', 'rc', 's1', 's2']


    
X = np.load(os.path.join(r"C:\Users\User\Desktop\高三專題\數據\ML_data", "ba.X.npy"))
y = np.load(os.path.join(r"C:\Users\User\Desktop\高三專題\數據\ML_data", "ba.y.npy"))

print(X.shape)  # (視窗數, 50, 欄位數)    
print(X[1990])    
print(y[1990])   # 第一個視窗的數據 (50, 欄位數)

(1991, 50, 24)
[[ 0.631   -0.111    0.107   ... -0.47588 -0.08649 -0.16453]
 [ 0.273   -0.2     -0.04    ... -0.55184 -0.34126 -0.26975]
 [-0.111   -0.355    0.059   ... -0.5586  -0.69368 -0.27383]
 ...
 [ 0.616    0.259   -0.009   ...  0.2858   1.7827  -0.12832]
 [ 0.636    0.283   -0.022   ...  0.08232  1.78998  0.12479]
 [ 0.675    0.328   -0.008   ...  0.04866  1.45292  0.43612]]
[-0.056 -0.065]


In [5]:
# 創所有linear用的數據

process_ml_data_linear()

已生成 ba: X=(1991, 24), y=(1991, 2)
已生成 bc: X=(1991, 24), y=(1991, 2)
已生成 ra: X=(1991, 24), y=(1991, 2)
已生成 rc: X=(1991, 24), y=(1991, 2)
已生成 s1: X=(1991, 24), y=(1991, 2)
已生成 s2: X=(1991, 24), y=(1991, 2)


In [9]:
experiment_name_list = ['ba', 'bc', 'ra', 'rc', 's1', 's2']

X = np.load(os.path.join(r"C:\Users\User\Desktop\高三專題\數據\ML\data", "ba.X_linear.npy"))
y = np.load(os.path.join(r"C:\Users\User\Desktop\高三專題\數據\ML\data", "ba.y_linear.npy"))

print(X.shape)  # (視窗數, 50, 欄位數)  
print(y.shape)  
print(X[0])    
print(y[0])   # 第一個視窗的數據 (50, 欄位數)

(1991, 24)
(1991, 2)
[ 0.189    0.191    0.045    0.44904 -1.26365  0.77008  0.274    1.201
 -0.659    1.18591  2.73193  1.06653  0.892    0.441   -0.53     0.69497
  1.23032  0.5513  -0.333    0.112   -0.587    0.05345  1.18925 -1.66817]
[-0.367 -0.189]
